# QLoRA: huấn luyện mở rộng
Một T4, Internet bật. 770 train / 114 validation, không có holdout. Chạy từng cell theo thứ tự; cell 4 chạy một epoch từ base mới, không nối tiếp smoke.


In [ ]:
from pathlib import Path
import hashlib, json, os, stat, zipfile, subprocess, sys, time
from datetime import datetime, timezone
os.environ.update(CUDA_VISIBLE_DEVICES='0', USE_TF='0', USE_FLAX='0', PYTHONDONTWRITEBYTECODE='1', TOKENIZERS_PARALLELISM='false')
EXPECTED = '8546d96bb335fa9697838a7ff3481152a69be2c4faef92af795d335e4e51101e'
EXPECTED_TRANSPORT = '3592dc81b09b7765040b79c8c1dcac6397b72f90aec0b256405616bd59b1c10a'
matches = [p for p in Path('/kaggle/input').rglob('training-runner.zip') if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED]
PACKAGE = Path('/kaggle/working/comparison-training-package-' + EXPECTED[:12])
if matches:
    assert len(matches) == 1, 'Duplicate matching ZIP inputs.'
    with zipfile.ZipFile(matches[0]) as z:
        names = z.namelist()
        assert len(names) == len(set(names))
        transport = z.read('bundle-manifest.json')
        assert hashlib.sha256(transport).hexdigest() == EXPECTED_TRANSPORT
        manifest = json.loads(transport)
        assert set(names) == set(manifest) | {'bundle-manifest.json'}
        assert all(not stat.S_ISLNK(i.external_attr >> 16) for i in z.infolist())
        payload = {name: z.read(name) for name in manifest}
else:
    manifests = [p for p in Path('/kaggle/input').rglob('bundle-manifest.json') if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_TRANSPORT]
    assert len(manifests) == 1, 'Add the exact runner-dev ZIP or its extracted Kaggle Dataset.'
    source = manifests[0].parent.resolve()
    manifest = json.loads(manifests[0].read_text())
    payload = {}
    for name in manifest:
        p = (source / name).resolve()
        assert p.is_relative_to(source) and not (source / name).is_symlink()
        payload[name] = p.read_bytes()
for name, raw in payload.items():
    target = (PACKAGE / name).resolve()
    assert target.is_relative_to(PACKAGE.resolve()) and not Path(name).is_absolute()
    assert chr(92) not in name and ':' not in name
    assert len(raw) == manifest[name]['bytes'] and hashlib.sha256(raw).hexdigest() == manifest[name]['sha256']
for name, raw in payload.items():
    target = PACKAGE / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).hexdigest() == manifest[name]['sha256']
    else:
        with target.open('xb') as f: f.write(raw)
PYTHON = Path('/kaggle/working/comparison-training-env/bin/python')
print('Training package authenticated: 770 train / 114 validation. Main locked.')

REQUIRE_EXISTING_SMOKE = True


In [ ]:
subprocess.run([sys.executable, '-B', '-m', 'scripts.comparison.training_kaggle', '--package', str(PACKAGE)], cwd=PACKAGE, check=True)


In [ ]:
def preserve_training_failure(run, console):
    archive = Path(str(run) + '-failed.zip')
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(console, 'console.log')
        for p in sorted(run.rglob('*')):
            if p.is_file(): z.write(p, p.relative_to(run).as_posix())
    print('Failure evidence:', archive, 'SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())

def train_run(mode):
    run = Path('/kaggle/working/comparison-training-runs') / (mode + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    run.parent.mkdir(parents=True, exist_ok=True)
    console = Path(str(run) + '-console.log')
    started = time.monotonic()
    with console.open('x', encoding='utf-8') as log:
        process = subprocess.Popen([str(PYTHON), '-B', '-m', 'scripts.comparison.training', '--package', str(PACKAGE), '--output', str(run), '--mode', mode], cwd=PACKAGE, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            try:
                phase = json.loads((run / 'phase.json').read_text())
            except (OSError, json.JSONDecodeError):
                phase = dict(phase='STARTING')
            print(str(int(time.monotonic()-started)) + 's | ' + json.dumps(phase), flush=True)
            time.sleep(15)
    if process.returncode:
        preserve_training_failure(run, console)
        raise RuntimeError('Training failed. Send failure ZIP; do not overwrite or retry silently.')
    audited = subprocess.run([str(PYTHON), '-B', '-m', 'scripts.comparison.training_audit', '--run', str(run), '--package', str(PACKAGE)], cwd=PACKAGE)
    if audited.returncode:
        preserve_training_failure(run, console)
        raise RuntimeError('Audit failed. Send failure ZIP; do not train again.')
    audit = json.loads(Path(str(run)+'-audit.json').read_text())
    print(json.dumps(audit, indent=2))
    print('Figures:', run / 'figures')
    return run, audit
OLD_SMOKE = Path('/kaggle/working/comparison-training-runs/smoke-20261112T184520250008Z')
OLD_PACKAGE = Path('/kaggle/working/comparison-training-package-c56f7d302c62')
if OLD_SMOKE.is_dir():
    assert OLD_PACKAGE.is_dir(), 'Keep the old package for authenticated recovery. Do not retrain.'
    expected_old_run = 'd00dc8a4027622cf62f3cd3091dc900baf018e96034f0700c06378e876105aaf'
    assert hashlib.sha256((OLD_SMOKE/'run-manifest.json').read_bytes()).hexdigest() == expected_old_run
    audited = subprocess.run([str(PYTHON), '-B', '-m', 'scripts.comparison.training_audit', '--run', str(OLD_SMOKE), '--package', str(OLD_PACKAGE)], cwd=PACKAGE)
    assert audited.returncode == 0, 'Recovery audit failed. Do not retrain; send output.'
    TRAIN_SMOKE_RUN = OLD_SMOKE
    TRAIN_SMOKE_AUDIT = json.loads(Path(str(OLD_SMOKE)+'-audit.json').read_text())
    print('Existing smoke recovered without training:', json.dumps(TRAIN_SMOKE_AUDIT, indent=2))
else:
    assert not REQUIRE_EXISTING_SMOKE, 'Completed smoke directory missing. Keep the session or restore its authenticated evidence; do not retrain automatically.'
    TRAIN_SMOKE_RUN, TRAIN_SMOKE_AUDIT = train_run('smoke')


In [ ]:
assert TRAIN_SMOKE_AUDIT['status'] == 'EXPANDED_TRAINING_SMOKE_PASSED'
FULL_RUN, FULL_AUDIT = train_run('full')


In [ ]:
assert FULL_AUDIT['status'] == 'EXPANDED_ONE_EPOCH_TRAINING_PASSED'
DEV_PACKAGE = Path(str(FULL_RUN) + '-dev-package')
subprocess.run([str(PYTHON), '-B', '-m', 'scripts.comparison.training_audit', '--run', str(FULL_RUN), '--package', str(PACKAGE), '--dev-package', str(DEV_PACKAGE)], cwd=PACKAGE, check=True)
TRAINING_PACKAGE = PACKAGE
PACKAGE = DEV_PACKAGE
def execute(mode):
    run = Path('/kaggle/working/comparison-runs') / (mode + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    run.parent.mkdir(parents=True, exist_ok=True)
    console = Path(str(run) + '-console.log')
    started = time.monotonic()
    with console.open('x', encoding='utf-8') as log:
        process = subprocess.Popen([str(PYTHON), '-B', '-m', 'scripts.comparison.runner', '--package', str(PACKAGE), '--output', str(run), '--mode', mode], cwd=PACKAGE, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            try:
                progress = json.loads((run / 'progress.json').read_text())
            except (OSError, json.JSONDecodeError):
                progress = dict(phase='STARTING', rows=0)
            print(str(int(time.monotonic()-started)) + 's | ' + json.dumps(progress), flush=True)
            time.sleep(15)
    audit = json.loads((run / 'audit.json').read_text()) if (run / 'audit.json').exists() else None
    archive = Path(str(run) + '.zip')
    with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(console, 'console.log')
        for p in sorted(run.rglob('*')):
            if p.is_file(): z.write(p, p.relative_to(run).as_posix())
    receipt = dict(archive=archive.name, sha256=hashlib.sha256(archive.read_bytes()).hexdigest(), size_bytes=archive.stat().st_size, process_exit_code=process.returncode, audit=audit)
    receipt_path = Path(str(run) + '-receipt.json')
    receipt_path.write_text(json.dumps(receipt, indent=2), encoding='utf-8')
    print(json.dumps(receipt, indent=2))
    print('ZIP:', archive, '\nReceipt:', receipt_path, '\nAudit:', run / 'audit.json')
    assert process.returncode == 0 and audit is not None, 'Run incomplete; the ZIP and receipt preserve available failure evidence.'
    return run, audit
SMOKE_RUN, SMOKE_AUDIT = execute('smoke')
assert SMOKE_AUDIT['failure_rows'] == 0
DEV_RUN, DEV_AUDIT = execute('dev')
